In [ ]:
#run refresh_trade_views to init arbitrage tables
#from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
#from domain.market_rows import MarketGoodRow, MarketTransactionRow
#import sqlite3
from db.refresh_trade_views import refresh_trade_views
from services.arbitrage_repo import top_arbitrage

# init arbitrage and trade views tables
refresh_trade_views(lookback_hours=48, create_schema=True)

# get highest arbitrage data
ta = top_arbitrage()
print(ta)

In [ ]:
ta.iloc[0].values.tolist()

In [ ]:
# 1. get fresh arbitrage data
update_arbi_data = refresh_trade_views(lookback_hours=48, create_schema=False)

# 2. calculate arbitrage trade profitability based on sales profit minus travel cost
 
# 3. if profitable*, calculate route to buy point from current location, navigate to buy point

# 4. buy cargo, navigate to sell point, sell cargo, repeat at step 1.

# * if no profitable arbitrage is available ship needs to switch to cargo transport from miners to profitable markets


In [21]:
# run refresh_trade_views to init arbitrage tables
from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
from domain.market_rows import MarketGoodRow, MarketTransactionRow
import sqlite3
from services.arbitrage_repo import top_arbitrage

# get highest arbitrage data

ta = top_arbitrage()

#print(ta.iloc[0,0])
arbi_trade_symbol = ta.iloc[0,0]
arbi_buy_wp = ta.iloc[0,1]
arbi_buy_price = ta.iloc[0,2]
arbi_sell_wp = ta.iloc[0,3]
arbi_sell_price = ta.iloc[0,4]

arbi_trade_symbol2 = ta.iloc[1,0]
arbi_buy_wp2 = ta.iloc[1,1]
arbi_buy_price2 = ta.iloc[1,2]
arbi_sell_wp2 = ta.iloc[1,3]
arbi_sell_price2 = ta.iloc[1,4]

arbi_trade_symbol3 = ta.iloc[2,0]
arbi_buy_wp3 = ta.iloc[2,1]
arbi_buy_price3 = ta.iloc[2,2]
arbi_sell_wp3 = ta.iloc[2,3]
arbi_sell_price3 = ta.iloc[2,4]

print("This is the trade symbol: ", arbi_trade_symbol)
print("This is the buy waypoint: ", arbi_buy_wp)
print("This is the buy price: ", arbi_buy_price)
print("This is the sell waypoint: ", arbi_sell_wp)
print("This is the sell price: ", arbi_sell_price)
print(ta)

This is the trade symbol:  ASSAULT_RIFLES
This is the buy waypoint:  X1-ZP92-E42
This is the buy price:  2356
This is the sell waypoint:  X1-ZP92-J57
This is the sell price:  4558
       trade_symbol buy_waypoint  buy_price sell_waypoint  sell_price  delta  \
0    ASSAULT_RIFLES  X1-ZP92-E42       2356   X1-ZP92-J57        4558   2202   
1   MICROPROCESSORS   X1-ZP92-A3       2245   X1-ZP92-D41        4277   2032   
2         EQUIPMENT  X1-ZP92-K86       2101   X1-ZP92-F48        3709   1608   
3              GOLD   X1-ZP92-B7        230   X1-ZP92-H53         363    133   
4       URANITE_ORE   X1-ZP92-B7        322   X1-ZP92-F48         403     81   
5              FUEL  X1-ZP92-G49         53   X1-ZP92-J57          68     15   
6  SILICON_CRYSTALS   X1-ZP92-B7         37    X1-ZP92-A3          45      8   
7   LIQUID_NITROGEN  X1-ZP92-C38         33   X1-ZP92-E43          39      6   
8   LIQUID_HYDROGEN  X1-ZP92-C38         27   X1-ZP92-F45          32      5   
9       QUARTZ_SAND 

In [16]:
from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
import math
import pandas as pd
from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_purchase_cargo,
    build_fleet_object
    )
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)

#Build all in-memory objects once (fleet_activity_obj, waypoints_ref_obj, waypoint_traits_obj, HQ)
world_state = await init_world_state(fleet_api, agents_api, systems_api)
from adapters.ships_activity_adapter import merge_activity_with_nav
from adapters.ships_specs_adapter import adapt_ships_specs_from_ship
from core_helpers import adapt_ships_activity_from_ship, upsert_many
from typing import Any, Dict, Iterable, List, Optional, Tuple
from runtime_support import call_sdk, unwrap_data
from dataclasses import dataclass, field
from domain.ships_activity import ShipsActivity
from domain.ships_specs import ShipsSpecs
from domain.ship_market import ShipMarketRow
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
@dataclass
class FleetState:
    """Local, easily-referenced state for your session."""
    # Activity & Specs keyed by ship symbol
    activities: Dict[str, ShipsActivity] = field(default_factory=dict)
    specs: Dict[str, ShipsSpecs] = field(default_factory=dict)

    # Shipyard listings cached by waypoint
    ship_market: Dict[str, List[ShipMarketRow]] = field(default_factory=dict)

    def ensure_activity(self, symbol: str) -> ShipsActivity:
        a = self.activities.get(symbol)
        if a is None:
            a = ShipsActivity(symbol=symbol)
            self.activities[symbol] = a
        return a

    def update_activity_from_nav(self, symbol: str, nav_dto: Any) -> ShipsActivity:
        current = self.ensure_activity(symbol)
        updated = merge_activity_with_nav(current, nav_dto)
        self.activities[symbol] = updated
        return updated

    def update_activity_from_refuel(self, symbol: str, resp_dto: Any) -> ShipsActivity:
        """Use refuel response to update fuel state in local activity."""
        current = self.ensure_activity(symbol)
        fuel_cur = getattr(getattr(resp_dto, "fuel", None), "current", None)
        fuel_cap = getattr(getattr(resp_dto, "fuel", None), "capacity", None)
        patched = current.model_copy(update={
            "fuel_current": fuel_cur if fuel_cur is not None else current.fuel_current,
            "fuel_capacity": fuel_cap if fuel_cap is not None else current.fuel_capacity,
        })
        self.activities[symbol] = patched
        return patched
async def local_fleet_state(fleet: FleetApi) -> FleetState:
    """Load ships once, build local state (activity + specs) and persist to DB."""
    resp = await call_sdk(fleet, "get_my_ships")
    ships: Iterable[Any] = unwrap_data(resp)
    ships = list(ships)
    if not ships:
        raise SystemExit("[FATAL] No ships returned; check token/agent.")

    activities = [adapt_ships_activity_from_ship(d) for d in ships]
    specs = [adapt_ships_specs_from_ship(d) for d in ships]
    state = FleetState(
        activities={a.symbol: a for a in activities if a.symbol},
        specs={s.symbol: s for s in specs if s.symbol},
    )
    return state

# --------- define ship roles -----------
state = await local_fleet_state(fleet_api)
print(state.specs)

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(state.specs, "COMMAND")
satellite = get_symbol_by_role(state.specs, "SATELLITE")
print(command_ship)

{'GLANK-1': ShipsSpecs(symbol='GLANK-1', role='COMMAND', frame_name='Frigate', frame_module_slots=8, frame_mounting_points=5, engine_name='Ion Drive II', speed=36, mounts=['MOUNT_SENSOR_ARRAY_II', 'MOUNT_GAS_SIPHON_II', 'MOUNT_MINING_LASER_II', 'MOUNT_SURVEYOR_II'], modules=['MODULE_CARGO_HOLD_II', 'MODULE_CREW_QUARTERS_I', 'MODULE_CREW_QUARTERS_I', 'MODULE_MINERAL_PROCESSOR_I', 'MODULE_GAS_PROCESSOR_I'], capacity=40), 'GLANK-2': ShipsSpecs(symbol='GLANK-2', role='SATELLITE', frame_name='Probe', frame_module_slots=0, frame_mounting_points=0, engine_name='Impulse Drive I', speed=9, mounts=[], modules=[], capacity=0)}
GLANK-1


In [17]:
from runtime_support import api_dock_ship, api_sell_cargo, api_purchase_cargo


In [23]:
from typing import Dict, List
from refuel_routing import Node
from domain.waypoint_trait import WaypointTraitRow

def build_nodes_from_traits_dict(
    traits_dict: Dict[str, List[WaypointTraitRow]],
    fuel_price_lookup: Dict[str, float] = None,
) -> List[Node]:
    """
    Convert a dict of {waypoint_symbol: [WaypointTraitRow, ...]} into Node objects.

    Args:
        traits_dict: mapping from waypoint_symbol -> list of WaypointTraitRow.
        fuel_price_lookup: optional {waypoint_symbol: price} for refining
                           Node.price (default None).

    Returns:
        List[Node]
    """
    nodes: List[Node] = []

    for symbol, trait_rows in traits_dict.items():
        if not trait_rows:
            continue  # skip empty

        # all rows for this waypoint share x,y,type
        first = trait_rows[0]
        x, y = float(first.x), float(first.y)

        # check if any trait is a MARKETPLACE
        has_marketplace = any(tr.trait_symbol == "MARKETPLACE" for tr in trait_rows)

        # build Node
        node = Node(
            symbol=symbol,
            x=x,
            y=y,
            has_fuel=has_marketplace,
            price=(fuel_price_lookup.get(symbol) if fuel_price_lookup else None),
        )
        nodes.append(node)

    return nodes

world_state = await init_world_state(fleet_api, agents_api, systems_api)

wps = world_state.traits.by_wp

nodes = build_nodes_from_traits_dict(wps)
from refuel_routing import (
    Node,
    build_candidates,
    build_reachability_graph,
    astar_by_time,
    compute_refuel_plan,
    corridor_polygon,
    plan_route_and_refuel,
)


plan_to_buy = plan_route_and_refuel(nodes, arbi_sell_wp, arbi_buy_wp, 400, 1, 1000, 400, 50, 0, 0, 0)
plan_to_sell = plan_route_and_refuel(nodes, arbi_buy_wp, arbi_sell_wp, 400, 1, 1000, 400, 50, 0, 0, 0)

plan_to_buy2 = plan_route_and_refuel(nodes, arbi_sell_wp2, arbi_buy_wp2, 400, 1, 1000, 400, 50, 0, 0, 0)
plan_to_sell2 = plan_route_and_refuel(nodes, arbi_buy_wp2, arbi_sell_wp2, 400, 1, 1000, 400, 50, 0, 0, 0)

plan_to_buy3 = plan_route_and_refuel(nodes, arbi_sell_wp3, arbi_buy_wp3, 400, 1, 1000, 400, 50, 0, 0, 0)
plan_to_sell3 = plan_route_and_refuel(nodes, arbi_buy_wp3, arbi_sell_wp3, 400, 1, 1000, 400, 50, 0, 0, 0)

In [ ]:
route_buy_point = plan_to_buy["path"]
route_sell_point = plan_to_sell["path"]

route_buy_point2 = plan_to_buy2["path"]
route_sell_point2 = plan_to_sell2["path"]

route_buy_point3 = plan_to_buy3["path"]
route_sell_point3 = plan_to_sell3["path"]

In [25]:
nav_resp = await api_get_ship_nav(fleet_api, command_ship)
nav_resp.waypoint_symbol

'X1-ZP92-D40'

In [30]:
async def trade_loop(
    fleet_api,
    command_ship,
    route_buy_point,
    route_sell_point,
    arbi_buy_wp,
    arbi_sell_wp,
    arbi_trade_symbol,
    units=20
):
    while True:  # repeat forever, or replace with a counter/condition

        # Navigate to buy waypoint(s)
        for rt in route_buy_point2:
            await api_navigate_ship(fleet_api, command_ship, rt)

        # Dock and buy
        await api_dock_ship(fleet_api, command_ship)
        await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp2, arbi_trade_symbol2, units)
        await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp2, arbi_trade_symbol2, units)

        # Navigate to sell waypoint(s)
        for rt in route_sell_point2:
            await api_navigate_ship(fleet_api, command_ship, rt)

        # Dock and sell
        await api_dock_ship(fleet_api, command_ship)
        await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp2, arbi_trade_symbol2, units)
        await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp2, arbi_trade_symbol2, units)


In [31]:
await trade_loop(fleet_api,
    command_ship,
    route_buy_point,
    route_sell_point,
    arbi_buy_wp,
    arbi_sell_wp,
    arbi_trade_symbol,
    units=20)

starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Prep complete
GLANK-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 111.891399
Arrived and ready
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
GLANK-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 65.885834
Arrived and ready
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
GLANK-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 65.876544
Arrived and read

BadRequestException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Access-Control-Allow-Origin': '*', 'Access-Control-Expose-Headers': 'Retry-After, X-RateLimit-Type, X-RateLimit-Limit-Burst, X-RateLimit-Limit-Per-Second, X-RateLimit-Remaining, X-RateLimit-Reset', 'Content-Length': '306', 'Content-Type': 'application/json; charset=utf-8', 'Date': 'Wed, 17 Sep 2025 19:00:07 GMT', 'Retry-After': '1', 'X-Ratelimit-Limit-Burst': '30', 'X-Ratelimit-Limit-Per-Second': '2', 'X-Ratelimit-Remaining': '0', 'X-Ratelimit-Reset': '2025-09-17T19:00:08.948Z', 'X-Ratelimit-Type': 'IP Address'})
HTTP response body: {"error":{"code":4600,"message":"Market purchase failed. Agent does not have sufficient credits to purchase 20 unit(s) of MICROPROCESSORS","data":{"agentCredits":28927,"totalPrice":245820,"tradeSymbol":"MICROPROCESSORS","units":20,"purchasePrice":12291},"requestId":"0199590c-6378-701d-92c6-c6efb7039e8c"}}
